# Enhanced Resilient Multi-Rate Data Processing
## Maximum Data Loading Success with High-Quality Preprocessing

This notebook combines three powerful approaches:
1. **Comprehensive Validation** - Pre-validate all datasets before processing
2. **Progressive Loading** - Process data in batches to avoid memory issues
3. **Resilient Error Handling** - Never fail completely, always return results

**Features:**
- ✅ Load ALL available data with maximum success rate
- ✅ High-quality audio and text preprocessing
- ✅ Memory-efficient batch processing
- ✅ Detailed logging and quality assessment
- ✅ Multi-rate audio processing (16kHz, 22kHz, 24kHz)
- ✅ Model-specific manifest generation

In [6]:
import os
from google.colab import drive
import sys
import logging

# Configure logging for detailed output
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Mount Google Drive
drive.mount('/content/drive')

# Create and set the working directory
BASE_PATH = '/content/drive/MyDrive/voice-lab-data/'
os.makedirs(BASE_PATH, exist_ok=True)
os.chdir(BASE_PATH)
print(f"🏠 Working directory set to: {os.getcwd()}")

# Add utilities to path for imports
sys.path.append('/content')

logger.info("Environment initialized successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🏠 Working directory set to: /content/drive/MyDrive/voice-lab-data


In [7]:
# Install required packages with specific versions for stability
# Install required packages with specific versions for stability
!pip install -q pandas numpy soundfile librosa tqdm pyarrow psutil torchcodec
!pip install -q datasets transformers torch torchaudio

print("📦 All packages installed successfully!")

📦 All packages installed successfully!


In [8]:
%%writefile enhanced_resilient_data_processor.py
#!/usr/bin/env python3
"""
Enhanced Resilient Multi-Rate Data Processing System
Combines Options 1, 2, and 3 for maximum data loading success with high-quality preprocessing.

Features:
- Progressive loading to avoid memory issues
- Comprehensive validation and error handling
- High-quality audio and text preprocessing
- Memory-efficient batch processing
- Detailed logging and recovery mechanisms
"""

import gc
import json
import time
import pickle
import re
import shutil
import unicodedata
import psutil
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any, Union, Iterator
from dataclasses import dataclass, field
from collections import defaultdict, Counter
import logging

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class MemoryManager:
    """Monitor and manage memory usage during processing."""

    def __init__(self, max_memory_percent=85):
        self.max_memory_percent = max_memory_percent
        self.process = psutil.Process()

    def check_memory(self) -> Dict[str, float]:
        """Check current memory usage."""
        memory_info = self.process.memory_info()
        virtual_memory = psutil.virtual_memory()

        return {
            'used_mb': memory_info.rss / 1024 / 1024,
            'available_mb': virtual_memory.available / 1024 / 1024,
            'percent_used': virtual_memory.percent,
            'is_critical': virtual_memory.percent > self.max_memory_percent
        }

    def force_gc(self):
        """Force garbage collection and log memory freed."""
        before = self.check_memory()
        gc.collect()
        after = self.check_memory()

        freed = before['used_mb'] - after['used_mb']
        if freed > 10:  # Only log if significant memory freed
            logger.info(f"🧹 Freed {freed:.1f}MB memory via GC")

        return freed

    def should_pause(self) -> bool:
        """Check if we should pause processing due to memory pressure."""
        memory = self.check_memory()
        return memory['is_critical']

@dataclass
class EnhancedMultiRateConfig:
    """Enhanced configuration with progressive loading parameters."""
    base_path: Path
    output_path: Path

    # Sample rate configurations
    sample_rates: Dict[str, int] = field(default_factory=lambda: {
        'asr_16k': 16000,      # ASR models: Whisper, Wav2Vec2, MMS, Moshi
        'tts_22k': 22050,      # TTS models: Coqui XTTS, MeloTTS
        'tts_24k': 24000       # High-quality TTS: OpenVoice, Kokoro, F5, ChatTTS
    })

    # Model mappings
    model_categories: Dict[str, List[str]] = field(default_factory=lambda: {
        'asr_16k': ['faster_whisper', 'wav2vec2', 'mms', 'moshi'],
        'tts_22k': ['coqui_xtts', 'melo_tts'],
        'tts_24k': ['openvoice', 'kokoro_tts', 'f5_tts', 'chattts_chatterbox', 'styletts2', 'meta_voice']
    })

    # Progressive loading parameters
    batch_size: int = 500  # Process this many samples at once
    max_samples_per_dataset: int = 5000  # Maximum samples from each dataset
    max_arrow_files_per_dataset: int = 10  # Limit Arrow files to prevent memory issues
    memory_check_interval: int = 100  # Check memory every N samples

    # Quality thresholds
    min_duration: float = 0.3  # Slightly lower for more data
    max_duration: float = 45.0  # Higher for more data
    min_text_length: int = 2
    max_text_length: int = 800
    target_db: float = -20.0

    # High-quality preprocessing parameters
    normalize_audio: bool = True
    apply_noise_reduction: bool = True
    enhance_text_processing: bool = True
    validate_linguistic_content: bool = True

    # Dataset configuration with priority ordering
    datasets_to_scan: List[Dict] = field(default_factory=lambda: [
        # High-priority datasets (process first)
        {
            "name": "twi_multi_train",
            "type": "arrow_sharded_audio",
            "path": "raw/michsethowusu/twi_multispeaker_audio_transcribed/train",
            "priority": 1,
            "max_samples": 3000
        },
        {
            "name": "twi_emotions_train",
            "type": "arrow_sharded_audio",
            "path": "raw/michsethowusu/twi-emotions-corpus/train",
            "priority": 1,
            "max_samples": 2000
        },
        # Medium-priority datasets
        {
            "name": "bibletts",
            "type": "bibletts",
            "path": "raw/bibletts",
            "priority": 2,
            "max_samples": 2000
        },
        {
            "name": "financial_inclusion",
            "type": "financial_inclusion",
            "path": "raw/financial_inclusion",
            "priority": 2,
            "max_samples": 1500
        },
        # Lower-priority datasets (related languages for transfer learning)
        {
            "name": "akuapem_multi_train",
            "type": "arrow_sharded_audio",
            "path": "raw/michsethowusu/akuapem_multispeaker_audio_transcribed/train",
            "priority": 3,
            "max_samples": 1000
        },
        {
            "name": "fante_multi_train",
            "type": "arrow_sharded_audio",
            "path": "raw/michsethowusu/fante_multispeaker_audio_transcribed/train",
            "priority": 3,
            "max_samples": 1000
        },
        {
            "name": "ga_multi_train",
            "type": "arrow_sharded_audio",
            "path": "raw/michsethowusu/ga_multispeaker_audio_transcribed/train",
            "priority": 3,
            "max_samples": 1000
        },
        # Test sets (smaller, for validation)
        {
            "name": "twi_multi_test",
            "type": "arrow_sharded_audio",
            "path": "raw/michsethowusu/twi_multispeaker_audio_transcribed/test",
            "priority": 4,
            "max_samples": 500
        },
        {
            "name": "twi_parallel_train",
            "type": "arrow_sharded_audio",
            "path": "raw/michsethowusu/twi-speech-text-parallel-multispeaker/train",
            "priority": 4,
            "max_samples": 1000
        }
    ])

class EnhancedDatasetValidator:
    """Comprehensive dataset validation with detailed reporting."""

    def __init__(self, base_path: Path):
        self.base_path = base_path
        self.validation_cache = {}

    def validate_all_datasets(self, dataset_configs: List[Dict]) -> Tuple[List[Dict], Dict]:
        """Return valid datasets and detailed validation report."""
        logger.info("🔍 Starting comprehensive dataset validation...")

        valid_configs = []
        validation_report = {
            'total_datasets': len(dataset_configs),
            'valid_datasets': 0,
            'invalid_datasets': 0,
            'estimated_total_samples': 0,
            'details': {}
        }

        # Sort by priority for optimal processing order
        sorted_configs = sorted(dataset_configs, key=lambda x: x.get('priority', 5))

        for config in tqdm(sorted_configs, desc="Validating datasets"):
            validation_result = self.validate_dataset_comprehensive(config)

            validation_report['details'][config['name']] = validation_result

            if validation_result['is_valid']:
                valid_configs.append(config)
                validation_report['valid_datasets'] += 1
                validation_report['estimated_total_samples'] += validation_result['estimated_samples']
                logger.info(f"✅ {config['name']}: {validation_result['estimated_samples']} samples ({validation_result['confidence']})")
            else:
                validation_report['invalid_datasets'] += 1
                logger.warning(f"❌ {config['name']}: {validation_result['error']}")

        # Log summary
        logger.info(f"""
📊 Validation Summary:
   Valid datasets: {validation_report['valid_datasets']}/{validation_report['total_datasets']}
   Estimated samples: {validation_report['estimated_total_samples']:,}
   Success rate: {validation_report['valid_datasets']/validation_report['total_datasets']*100:.1f}%
        """)

        return valid_configs, validation_report

    def validate_dataset_comprehensive(self, config: Dict) -> Dict:
        """Comprehensive validation with detailed analysis."""
        dataset_path = self.base_path / config['path']

        result = {
            'is_valid': False,
            'error': '',
            'estimated_samples': 0,
            'confidence': 'unknown',
            'file_count': 0,
            'total_size_mb': 0,
            'sample_files_checked': 0
        }

        if not dataset_path.exists():
            result['error'] = 'Path does not exist'
            return result

        try:
            if config['type'] == 'arrow_sharded_audio':
                return self._validate_arrow_comprehensive(dataset_path, config)
            elif config['type'] == 'bibletts':
                return self._validate_bibletts_comprehensive(dataset_path, config)
            elif config['type'] == 'financial_inclusion':
                return self._validate_financial_comprehensive(dataset_path, config)
            else:
                result['error'] = f"Unknown dataset type: {config['type']}"
                return result

        except Exception as e:
            result['error'] = f"Validation failed: {str(e)[:100]}"
            return result

    def _validate_arrow_comprehensive(self, dataset_path: Path, config: Dict) -> Dict:
        """Comprehensive Arrow dataset validation."""
        result = {
            'is_valid': False,
            'error': '',
            'estimated_samples': 0,
            'confidence': 'unknown',
            'file_count': 0,
            'total_size_mb': 0,
            'sample_files_checked': 0
        }

        # Find Arrow files
        arrow_files = sorted(list(dataset_path.glob('data-*.arrow')))
        if not arrow_files:
            arrow_files = sorted(list(dataset_path.glob('*.arrow')))

        if not arrow_files:
            result['error'] = 'No Arrow files found'
            return result

        result['file_count'] = len(arrow_files)

        # Calculate total size
        total_size = sum(f.stat().st_size for f in arrow_files)
        result['total_size_mb'] = total_size / 1024 / 1024

        # Validate sample of files
        files_to_check = min(3, len(arrow_files))
        valid_files = 0
        total_samples = 0

        for arrow_file in arrow_files[:files_to_check]:
            try:
                # Try reading file
                try:
                    df = pd.read_feather(arrow_file)
                except:
                    import pyarrow as pa
                    table = pa.ipc.RecordBatchFileReader(pa.memory_map(str(arrow_file))).read_all()
                    df = table.to_pandas()

                # Check required columns
                # Check required columns (Updated with correct names)
                text_col = self._find_column(df.columns, ['transcription', 'transcript', 'text', 'sentence'])
                audio_col = self._find_column(df.columns, ['audio_path', 'audio', 'speech'])

                if text_col and audio_col:
                    # Validate sample data
                    sample_count = len(df)
                    if sample_count > 0:
                        # Check first few rows for data quality
                        sample_row = df.iloc[0]
                        if (isinstance(sample_row[text_col], str) and
                            len(sample_row[text_col].strip()) > 0 and
                            sample_row[audio_col] is not None):

                            valid_files += 1
                            total_samples += sample_count

                result['sample_files_checked'] += 1

            except Exception as e:
                logger.debug(f"Failed to validate {arrow_file.name}: {e}")
                continue

        # Determine result
        if valid_files > 0:
            # Estimate total samples
            avg_samples_per_file = total_samples / valid_files
            estimated_total = int(avg_samples_per_file * len(arrow_files))

            # Apply dataset limit if specified
            max_samples = config.get('max_samples', estimated_total)
            result['estimated_samples'] = min(estimated_total, max_samples)

            result['is_valid'] = True
            result['confidence'] = 'high' if valid_files == files_to_check else 'medium'
        else:
            result['error'] = 'No valid data found in sample files'

        return result

    def _validate_bibletts_comprehensive(self, dataset_path: Path, config: Dict) -> Dict:
        """Comprehensive bibletts validation."""
        result = {
            'is_valid': False,
            'error': '',
            'estimated_samples': 0,
            'confidence': 'unknown',
            'file_count': 0,
            'total_size_mb': 0,
            'sample_files_checked': 0
        }

        # Find audio files
        audio_extensions = ['.wav', '.mp3', '.flac', '.ogg']
        audio_files = []
        for ext in audio_extensions:
            audio_files.extend(dataset_path.rglob(f'*{ext}'))

        # Filter out system files
        audio_files = [f for f in audio_files if '__MACOSX' not in str(f)]

        if not audio_files:
            result['error'] = 'No audio files found'
            return result

        result['file_count'] = len(audio_files)

        # Check sample files
        files_to_check = min(10, len(audio_files))
        valid_pairs = 0

        for audio_file in audio_files[:files_to_check]:
            text_file = audio_file.with_suffix('.txt')
            if text_file.exists():
                try:
                    text = text_file.read_text(encoding='utf-8').strip()
                    if len(text) > 2:  # Basic quality check
                        valid_pairs += 1
                except:
                    pass
            result['sample_files_checked'] += 1

        if valid_pairs > 0:
            # Estimate total valid pairs
            valid_ratio = valid_pairs / files_to_check
            estimated_total = int(len(audio_files) * valid_ratio)

            max_samples = config.get('max_samples', estimated_total)
            result['estimated_samples'] = min(estimated_total, max_samples)

            result['is_valid'] = True
            result['confidence'] = 'high' if valid_ratio > 0.8 else 'medium'
        else:
            result['error'] = 'No valid audio-text pairs found'

        return result

    def _validate_financial_comprehensive(self, dataset_path: Path, config: Dict) -> Dict:
        """Comprehensive financial inclusion validation."""
        result = {
            'is_valid': False,
            'error': '',
            'estimated_samples': 0,
            'confidence': 'unknown',
            'file_count': 0,
            'total_size_mb': 0,
            'sample_files_checked': 0
        }

        # Look for audio directories and manifest files
        audio_dirs = list(dataset_path.rglob('audios'))
        manifest_files = list(dataset_path.rglob('data.csv'))

        if not (audio_dirs and manifest_files):
            result['error'] = 'Missing audio directories or manifest files'
            return result

        # Check sample manifest
        valid_manifests = 0
        total_samples = 0

        for manifest_file in manifest_files[:3]:  # Check up to 3 manifests
            try:
                df = pd.read_csv(manifest_file, on_bad_lines='skip')
                if len(df) > 0:
                    path_col = self._find_column(df.columns, ['path', 'audio', 'filename'])
                    text_col = self._find_column(df.columns, ['text', 'transcript'])

                    if path_col and text_col:
                        # Count non-empty text entries
                        valid_entries = df[df[text_col].str.len() > 2].shape[0]
                        if valid_entries > 0:
                            valid_manifests += 1
                            total_samples += valid_entries

                result['sample_files_checked'] += 1
            except:
                pass

        if valid_manifests > 0:
            max_samples = config.get('max_samples', total_samples)
            result['estimated_samples'] = min(total_samples, max_samples)
            result['is_valid'] = True
            result['confidence'] = 'medium'
        else:
            result['error'] = 'No valid manifests found'

        return result

    def _find_column(self, columns: List[str], candidates: List[str]) -> Optional[str]:
        """Find first matching column from candidates."""
        return next((c for c in candidates if c in columns), None)

class ProgressiveResilientDataLoader:
    """Progressive data loader with comprehensive error handling and memory management."""

    def __init__(self, config: EnhancedMultiRateConfig):
        self.config = config
        self.memory_manager = MemoryManager()
        self.failed_datasets = []
        self.partial_loads = {}
        self.processing_stats = {
            'total_samples_attempted': 0,
            'total_samples_successful': 0,
            'datasets_processed': 0,
            'memory_cleanups': 0
        }

        # Initialize text processor
        from enhanced_text_processor import EnhancedTwiTextProcessor
        self.text_processor = EnhancedTwiTextProcessor()

    def load_all_datasets_progressive(self, validated_configs: List[Dict]) -> pd.DataFrame:
        """Load all datasets progressively with memory management."""
        logger.info("🚀 Starting progressive dataset loading...")

        all_samples = []

        for dataset_config in tqdm(validated_configs, desc="Loading datasets"):
            try:
                logger.info(f"📂 Loading {dataset_config['name']} (Priority {dataset_config.get('priority', 'N/A')})")

                # Memory check before each dataset
                memory_status = self.memory_manager.check_memory()
                logger.info(f"💾 Memory: {memory_status['used_mb']:.1f}MB used ({memory_status['percent_used']:.1f}%)")

                if memory_status['is_critical']:
                    logger.warning("⚠️ Memory critical - forcing cleanup")
                    self.memory_manager.force_gc()
                    self.processing_stats['memory_cleanups'] += 1

                # Load dataset in batches
                dataset_samples = self._load_dataset_progressive(dataset_config)

                if dataset_samples:
                    all_samples.extend(dataset_samples)
                    self.processing_stats['datasets_processed'] += 1
                    logger.info(f"✅ Loaded {len(dataset_samples)} samples from {dataset_config['name']}")
                else:
                    self.failed_datasets.append(dataset_config['name'])
                    logger.warning(f"⚠️ No samples loaded from {dataset_config['name']}")

                # Periodic cleanup
                if len(all_samples) % 1000 == 0 and len(all_samples) > 0:
                    self.memory_manager.force_gc()

            except Exception as e:
                self.failed_datasets.append(dataset_config['name'])
                logger.error(f"❌ Failed to load {dataset_config['name']}: {e}")
                continue

        # Create final DataFrame
        if all_samples:
            logger.info("📊 Creating final dataset DataFrame...")
            final_df = pd.DataFrame(all_samples)

            # Add comprehensive metadata
            final_df = self._add_comprehensive_metadata(final_df)

            self.processing_stats['total_samples_successful'] = len(final_df)

            # Log final statistics
            self._log_processing_summary(final_df)

            return final_df
        else:
            logger.warning("🚨 No samples loaded from any dataset - creating minimal test dataset")
            return self._create_enhanced_dummy_dataset()

    def _load_dataset_progressive(self, dataset_config: Dict) -> List[Dict]:
        """Load single dataset progressively."""
        dataset_path = self.config.base_path / dataset_config['path']
        max_samples = dataset_config.get('max_samples', self.config.max_samples_per_dataset)

        if dataset_config['type'] == 'arrow_sharded_audio':
            return self._load_arrow_progressive(dataset_path, dataset_config['name'], max_samples)
        elif dataset_config['type'] == 'bibletts':
            return self._load_bibletts_progressive(dataset_path, dataset_config['name'], max_samples)
        elif dataset_config['type'] == 'financial_inclusion':
            return self._load_financial_progressive(dataset_path, dataset_config['name'], max_samples)
        else:
            logger.error(f"Unknown dataset type: {dataset_config['type']}")
            return []

    def _load_arrow_progressive(self, dataset_path: Path, dataset_name: str, max_samples: int) -> List[Dict]:
        """Load Arrow dataset progressively with memory management."""
        arrow_files = sorted(list(dataset_path.glob('data-*.arrow')))
        if not arrow_files:
            arrow_files = sorted(list(dataset_path.glob('*.arrow')))

        if not arrow_files:
            logger.warning(f"No Arrow files found in {dataset_path}")
            return []

        # Limit number of files to prevent memory issues
        arrow_files = arrow_files[:self.config.max_arrow_files_per_dataset]

        samples = []
        samples_per_file = max_samples // len(arrow_files)

        for arrow_file in tqdm(arrow_files, desc=f"Processing {dataset_name}", leave=False):
            try:
                # Load file
                df = self._load_arrow_file_safe(arrow_file)
                if df is None:
                    continue


                                # Find columns (Updated with correct names)
                text_col = self._find_column_flexible(df.columns, ['transcription', 'transcript', 'text', 'sentence'])
                audio_col = self._find_column_flexible(df.columns, ['audio_path', 'audio', 'speech'])


                speaker_col = self._find_column_flexible(df.columns, ['speaker_id', 'client_id'])

                if not (text_col and audio_col):
                    logger.warning(f"Missing required columns in {arrow_file.name}")
                    continue

                # Process in batches to manage memory
                batch_samples = self._process_arrow_batches(
                    df, text_col, audio_col, speaker_col,
                    dataset_name, samples_per_file
                )

                samples.extend(batch_samples)

                # Clear dataframe from memory
                del df
                if len(samples) % 500 == 0:
                    self.memory_manager.force_gc()

                # Stop if we have enough samples
                if len(samples) >= max_samples:
                    break

            except Exception as e:
                logger.warning(f"Failed to process {arrow_file.name}: {e}")
                continue

        return samples[:max_samples]  # Ensure we don't exceed limit

    def _load_arrow_file_safe(self, arrow_file: Path) -> Optional[pd.DataFrame]:
        """Safely load Arrow file with fallback methods."""
        try:
            # Method 1: pandas
            return pd.read_feather(arrow_file)
        except:
            try:
                # Method 2: pyarrow
                import pyarrow as pa
                table = pa.ipc.RecordBatchFileReader(pa.memory_map(str(arrow_file))).read_all()
                return table.to_pandas()
            except:
                return None

    def _process_arrow_batches(self, df: pd.DataFrame, text_col: str, audio_col: str,
                              speaker_col: Optional[str], dataset_name: str, max_samples: int) -> List[Dict]:
        """Process Arrow DataFrame in memory-efficient batches."""
        samples = []
        batch_size = min(self.config.batch_size, len(df))

        # Create temporary audio directory
        temp_audio_dir = self.config.output_path / 'temp_audio' / dataset_name
        temp_audio_dir.mkdir(parents=True, exist_ok=True)

        for start_idx in range(0, min(len(df), max_samples), batch_size):
            end_idx = min(start_idx + batch_size, len(df), max_samples)
            batch_df = df.iloc[start_idx:end_idx]

            for idx, row in batch_df.iterrows():
                try:
                    # Validate data
                    audio_data = row.get(audio_col)
                    text = row.get(text_col)

                    if not (audio_data and isinstance(text, str) and len(text.strip()) > 1):
                        continue

                    # Extract audio
                    audio_array = np.array(audio_data['array'], dtype=np.float32)
                    sampling_rate = audio_data['sampling_rate']

                    # Quick quality check
                    duration = len(audio_array) / sampling_rate
                    if not (self.config.min_duration <= duration <= self.config.max_duration):
                        continue

                    # Save temporary audio file
                    temp_filename = f"{dataset_name}_{idx}.wav"
                    temp_filepath = temp_audio_dir / temp_filename
                    sf.write(temp_filepath, audio_array, sampling_rate)

                    # Process text
                    processed_text = self.text_processor.process_text(text, expand_numbers=True)

                    # Create sample record
                    speaker_id = row.get(speaker_col, f"{dataset_name}_speaker_{idx}")

                    sample = {
                        'audio_path': str(temp_filepath),
                        'text': text,
                        'processed_text': processed_text['processed'],
                        'speaker_id': f"{dataset_name}_{speaker_id}",
                        'language': 'tw',
                        'dataset': dataset_name,
                        'dialect': 'michset',
                        'duration': duration,
                        'sample_rate': sampling_rate,
                        'text_length': len(text),
                        'word_count': len(text.split()),
                        'has_twi_chars': processed_text['has_twi_chars'],
                        'quality_score': self._calculate_quality_score(text, duration)
                    }

                    samples.append(sample)
                    self.processing_stats['total_samples_attempted'] += 1

                    # Memory check
                    if len(samples) % self.config.memory_check_interval == 0:
                        if self.memory_manager.should_pause():
                            self.memory_manager.force_gc()

                except Exception as e:
                    logger.debug(f"Failed to process sample {idx}: {e}")
                    continue

            # Clear batch from memory
            del batch_df

        return samples

    def _load_bibletts_progressive(self, dataset_path: Path, dataset_name: str, max_samples: int) -> List[Dict]:
        """Load bibletts dataset progressively."""
        audio_extensions = ['.wav', '.mp3', '.flac', '.ogg']
        audio_files = []

        for ext in audio_extensions:
            audio_files.extend(dataset_path.rglob(f'*{ext}'))

        # Filter system files
        audio_files = [f for f in audio_files if '__MACOSX' not in str(f)]
        audio_files = audio_files[:max_samples]  # Limit files

        samples = []

        for audio_file in tqdm(audio_files, desc=f"Processing {dataset_name}", leave=False):
            try:
                text_file = audio_file.with_suffix('.txt')
                if not text_file.exists():
                    continue

                text = text_file.read_text(encoding='utf-8', errors='ignore').strip()
                if len(text) < 2:
                    continue

                # Get audio info
                try:
                    info = sf.info(str(audio_file))
                    duration = info.duration

                    if not (self.config.min_duration <= duration <= self.config.max_duration):
                        continue

                except:
                    continue

                # Process text
                processed_text = self.text_processor.process_text(text, expand_numbers=True)

                # Determine dialect and speaker
                parts = audio_file.parts
                dialect_name = 'bibletts'
                for part in parts:
                    if any(d in part.lower() for d in ['asante', 'akuapem', 'fante']):
                        dialect_name = part.lower()
                        break

                speaker_id = f"{dataset_name}_{dialect_name}_{audio_file.parent.name}"

                sample = {
                    'audio_path': str(audio_file),
                    'text': text,
                    'processed_text': processed_text['processed'],
                    'speaker_id': speaker_id,
                    'language': 'tw',
                    'dataset': dataset_name,
                    'dialect': dialect_name,
                    'duration': duration,
                    'sample_rate': info.samplerate,
                    'text_length': len(text),
                    'word_count': len(text.split()),
                    'has_twi_chars': processed_text['has_twi_chars'],
                    'quality_score': self._calculate_quality_score(text, duration)
                }

                samples.append(sample)
                self.processing_stats['total_samples_attempted'] += 1

                # Memory management
                if len(samples) % self.config.memory_check_interval == 0:
                    if self.memory_manager.should_pause():
                        self.memory_manager.force_gc()

            except Exception as e:
                logger.debug(f"Failed to process {audio_file}: {e}")
                continue

        return samples

    def _load_financial_progressive(self, dataset_path: Path, dataset_name: str, max_samples: int) -> List[Dict]:
        """Load financial inclusion dataset progressively."""
        audio_extensions = ['.wav', '.mp3', '.ogg', '.flac']
        audio_files = []

        for ext in audio_extensions:
            audio_files.extend(dataset_path.rglob(f'**/audios/*{ext}'))

        audio_files = [f for f in audio_files if '__MACOSX' not in str(f)]
        audio_files = audio_files[:max_samples]

        # Cache manifest files
        manifest_cache = {}
        samples = []

        for audio_file in tqdm(audio_files, desc=f"Processing {dataset_name}", leave=False):
            try:
                manifest_path = audio_file.parent.parent / 'data.csv'
                if not manifest_path.exists():
                    continue

                # Load manifest if not cached
                if str(manifest_path) not in manifest_cache:
                    try:
                        df = pd.read_csv(manifest_path, on_bad_lines='skip')
                        path_col = self._find_column_flexible(df.columns, ['path', 'audio', 'filename'])
                        text_col = self._find_column_flexible(df.columns, ['text', 'transcript'])

                        if path_col and text_col:
                            manifest_cache[str(manifest_path)] = df.set_index(path_col)[text_col].to_dict()
                        else:
                            manifest_cache[str(manifest_path)] = {}
                    except:
                        manifest_cache[str(manifest_path)] = {}

                # Get transcript
                transcripts = manifest_cache.get(str(manifest_path), {})
                transcript = (transcripts.get(audio_file.name) or
                            transcripts.get(f"audios/{audio_file.name}"))

                if not (transcript and isinstance(transcript, str) and len(transcript.strip()) > 1):
                    continue

                # Get audio info
                try:
                    info = sf.info(str(audio_file))
                    duration = info.duration

                    if not (self.config.min_duration <= duration <= self.config.max_duration):
                        continue
                except:
                    continue

                # Process text
                processed_text = self.text_processor.process_text(transcript, expand_numbers=True)

                speaker_id = f"{dataset_name}_{audio_file.parent.parent.name}"

                sample = {
                    'audio_path': str(audio_file),
                    'text': transcript,
                    'processed_text': processed_text['processed'],
                    'speaker_id': speaker_id,
                    'language': 'tw',
                    'dataset': dataset_name,
                    'dialect': 'financial',
                    'duration': duration,
                    'sample_rate': info.samplerate,
                    'text_length': len(transcript),
                    'word_count': len(transcript.split()),
                    'has_twi_chars': processed_text['has_twi_chars'],
                    'quality_score': self._calculate_quality_score(transcript, duration)
                }

                samples.append(sample)
                self.processing_stats['total_samples_attempted'] += 1

            except Exception as e:
                logger.debug(f"Failed to process {audio_file}: {e}")
                continue

        return samples

    def _find_column_flexible(self, columns: List[str], candidates: List[str]) -> Optional[str]:
        """Flexibly find column with partial matching."""
        # Exact match first
        for candidate in candidates:
            if candidate in columns:
                return candidate

        # Partial match
        for candidate in candidates:
            for col in columns:
                if candidate.lower() in col.lower():
                    return col

        return None

    def _calculate_quality_score(self, text: str, duration: float) -> float:
        """Calculate quality score for sample."""
        score = 5.0  # Base score

        # Text quality factors
        if len(text) > 50:
            score += 1.0
        if any(c in text for c in 'ɛɔ'):  # Twi-specific characters
            score += 1.5
        if re.search(r'[.!?]', text):  # Proper punctuation
            score += 0.5

        # Duration factors
        if 2.0 <= duration <= 15.0:  # Optimal duration
            score += 1.0
        elif duration < 1.0 or duration > 20.0:
            score -= 1.0

        # Text/duration ratio
        words_per_second = len(text.split()) / duration
        if 2.0 <= words_per_second <= 6.0:  # Natural speaking rate
            score += 1.0

        return min(10.0, max(1.0, score))

    def _add_comprehensive_metadata(self, df: pd.DataFrame) -> pd.DataFrame:
        """Add comprehensive metadata to the dataset."""
        logger.info("📊 Adding comprehensive metadata...")

        # Audio exists check
        df['audio_exists'] = df['audio_path'].apply(lambda x: Path(x).exists())

        # Channels info for existing files
        channels_info = []
        for audio_path in tqdm(df['audio_path'], desc="Checking audio files", leave=False):
            try:
                if Path(audio_path).exists():
                    info = sf.info(audio_path)
                    channels_info.append(info.channels)
                else:
                    channels_info.append(0)
            except:
                channels_info.append(0)

        df['channels'] = channels_info

        # Quality filters
        df['passes_quality_filter'] = (
            (df['audio_exists']) &
            (df['duration'] >= self.config.min_duration) &
            (df['duration'] <= self.config.max_duration) &
            (df['text_length'] >= self.config.min_text_length) &
            (df['text_length'] <= self.config.max_text_length) &
            (df['quality_score'] >= 3.0)
        )

        return df

    def _create_enhanced_dummy_dataset(self) -> pd.DataFrame:
        """Create enhanced dummy dataset with comprehensive metadata."""
        logger.info("🎭 Creating enhanced dummy dataset...")

        dummy_dir = self.config.output_path / 'dummy_audio'
        dummy_dir.mkdir(parents=True, exist_ok=True)

        # Enhanced Twi samples with variety
        samples = [
            ("Wo ho te sɛn? Me ho yɛ paa.", "greeting", "speaker_001"),
            ("Me ma wo akye. Nante yie.", "greeting", "speaker_002"),
            ("Me da wo ase pii sɛ woboa me.", "gratitude", "speaker_003"),
            ("Twi kasa yɛ fɛ papa. Ɛyɛ anigyesɛm sɛ mesua.", "language_learning", "speaker_004"),
            ("Obiara pɛ sɛ ɔnya ahotɔ wɔ n'ankasa kasa mu.", "cultural_expression", "speaker_005"),
            ("Yɛn nkurɔfoɔ nyinaa yɛ baako wɔ amoa kɛseɛ yi mu.", "unity", "speaker_006"),
            ("Adwo, ɔkyena yɛbɛhyia bio wɔ sukuu mu.", "daily_conversation", "speaker_007"),
            ("Aboa kɛseɛ yi mu no, yɛde anigyeɛ kyekyɛ yɛn ho.", "community", "speaker_008")
        ]

        dummy_data = []

        for i, (text, context, speaker) in enumerate(samples):
            audio_file = dummy_dir / f"{speaker}_{i:03d}.wav"

            if not audio_file.exists():
                # Create varied length audio (2-8 seconds)
                duration = 2.0 + (i % 6)
                dummy_audio = np.random.normal(0, 0.05, int(22050 * duration)).astype(np.float32)
                # Add slight amplitude variation to simulate speech
                envelope = np.sin(np.linspace(0, np.pi * duration, len(dummy_audio))) * 0.5 + 0.5
                dummy_audio = dummy_audio * envelope
                sf.write(audio_file, dummy_audio, 22050)

            # Get actual file info
            info = sf.info(str(audio_file))
            processed_text = self.text_processor.process_text(text, expand_numbers=True)

            dummy_data.append({
                'audio_path': str(audio_file),
                'text': text,
                'processed_text': processed_text['processed'],
                'speaker_id': speaker,
                'language': 'tw',
                'dataset': 'dummy_enhanced',
                'dialect': 'test',
                'context': context,
                'duration': info.duration,
                'sample_rate': info.samplerate,
                'channels': info.channels,
                'audio_exists': True,
                'text_length': len(text),
                'word_count': len(text.split()),
                'has_twi_chars': processed_text['has_twi_chars'],
                'quality_score': self._calculate_quality_score(text, info.duration),
                'passes_quality_filter': True
            })

        return pd.DataFrame(dummy_data)

    def _log_processing_summary(self, df: pd.DataFrame):
        """Log comprehensive processing summary."""
        quality_df = df[df['passes_quality_filter']]

        logger.info(f"""
🎉 PROCESSING COMPLETE!
===============================================
📊 Dataset Statistics:
   Total samples loaded: {len(df):,}
   High-quality samples: {len(quality_df):,}
   Quality retention rate: {len(quality_df)/len(df)*100:.1f}%

📂 Dataset Breakdown:
   Datasets processed: {self.processing_stats['datasets_processed']}
   Failed datasets: {len(self.failed_datasets)}
   Success rate: {self.processing_stats['datasets_processed']/(self.processing_stats['datasets_processed']+len(self.failed_datasets))*100:.1f}%

🎵 Audio Statistics:
   Total duration: {df['duration'].sum()/3600:.1f} hours
   Quality duration: {quality_df['duration'].sum()/3600:.1f} hours
   Average duration: {df['duration'].mean():.1f}s
   Duration range: {df['duration'].min():.1f}s - {df['duration'].max():.1f}s

📝 Text Statistics:
   Average text length: {df['text_length'].mean():.0f} chars
   Average word count: {df['word_count'].mean():.1f} words
   Twi characters present: {df['has_twi_chars'].sum()} samples ({df['has_twi_chars'].mean()*100:.1f}%)

👥 Speaker Statistics:
   Unique speakers: {df['speaker_id'].nunique():,}
   Languages: {list(df['language'].unique())}
   Dialects: {list(df['dialect'].unique())}

💾 Memory Management:
   Memory cleanups performed: {self.processing_stats['memory_cleanups']}
   Final memory usage: {self.memory_manager.check_memory()['used_mb']:.1f}MB

{"❌ Failed Datasets: " + str(self.failed_datasets) if self.failed_datasets else "✅ All datasets loaded successfully!"}
===============================================
        """)

print("✅ Enhanced resilient data processing system loaded successfully!")
print("🚀 Features: Progressive loading + Comprehensive validation + Memory management + High-quality preprocessing")

print("✅ Enhanced resilient data processor loaded!")

Overwriting enhanced_resilient_data_processor.py


In [9]:
%%writefile enhanced_text_processor.py
#!/usr/bin/env python3
"""
Enhanced Twi Text Processor
High-quality text preprocessing for Twi language with comprehensive linguistic support.

Features:
- Comprehensive Twi number conversion
- Advanced text normalization
- Cultural context preservation
- Dialect-aware processing
- Quality assessment and validation
"""

import re
import unicodedata
import logging
from typing import Dict, List, Optional, Tuple, Set
from dataclasses import dataclass
from collections import defaultdict

logger = logging.getLogger(__name__)

@dataclass
class TwiLinguisticFeatures:
    """Twi-specific linguistic features and patterns."""

    # Twi-specific characters
    twi_vowels: Set[str] = frozenset({'a', 'e', 'ɛ', 'i', 'o', 'ɔ', 'u'})
    twi_special_chars: Set[str] = frozenset({'ɛ', 'Ɛ', 'ɔ', 'Ɔ', 'ã', 'ẽ', 'ĩ', 'õ', 'ũ'})

    # Common Twi prefixes and suffixes
    common_prefixes: Set[str] = frozenset({'a', 'o', 'ɔ', 'me', 'wo', 'ɛ', 're', 'be'})
    common_suffixes: Set[str] = frozenset({'e', 'ɛ', 'i', 'o', 'ɔ', 'u', 'a', 'foɔ', 'nom'})

    # Tone markers (though not always written)
    tone_markers: Set[str] = frozenset({'́', '̀', '̂', '̌', '̄'})

    # Common Twi words for validation
    common_words: Set[str] = frozenset({
        'wo', 'me', 'ɔ', 'yɛ', 'ne', 'wɔ', 'na', 'sɛ', 'mu', 'ho', 'no', 'de', 'a', 'ɛ',
        'baako', 'mmienu', 'mmiɛnsa', 'nnan', 'nnum', 'nsia', 'nson', 'nwɔtwe', 'nkron',
        'akye', 'maakye', 'adwo', 'anadwo', 'ɔkyena', 'nnɛ', 'nante', 'yie', 'ase', 'paa',
        'kasa', 'dwom', 'nsɛm', 'amoa', 'ɔman', 'fie', 'sukuu', 'adwuma', 'aduane'
    })

class TwiNumberProcessor:
    """Advanced Twi number conversion system."""

    def __init__(self):
        # Basic numbers 0-10
        self.units = {
            0: 'hwee', 1: 'baako', 2: 'mmienu', 3: 'mmiɛnsa',
            4: 'nnan', 5: 'nnum', 6: 'nsia', 7: 'nson',
            8: 'nwɔtwe', 9: 'nkron', 10: 'edu'
        }

        # Numbers 11-19
        self.teens = {
            11: 'dubaako', 12: 'dummienu', 13: 'dummiɛnsa',
            14: 'dunan', 15: 'dunum', 16: 'dunsia',
            17: 'dunson', 18: 'dunwɔtwe', 19: 'dunkron'
        }

        # Tens 20-90
        self.tens = {
            20: 'aduonu', 30: 'aduasa', 40: 'aduanan',
            50: 'aduonum', 60: 'aduosia', 70: 'aduɔson',
            80: 'aduɔwɔtwe', 90: 'aduɔkron'
        }

        # Scale units
        self.scale = {
            100: 'ɔha', 1000: 'apem', 1000000: 'ɔpepem'
        }

        # Ordinal numbers
        self.ordinals = {
            1: 'deɛ ɛdi kan', 2: 'deɛ ɛtɔ so mmienu', 3: 'deɛ ɛtɔ so mmiɛnsa',
            4: 'deɛ ɛtɔ so nnan', 5: 'deɛ ɛtɔ so nnum'
        }

    def convert_number(self, num: int, ordinal: bool = False) -> str:
        """Convert integer to Twi words."""
        if ordinal and num in self.ordinals:
            return self.ordinals[num]

        if num == 0:
            return self.units[0]

        if num < 0:
            return f"negative {self.convert_number(-num)}"

        # Handle basic numbers
        if num <= 10:
            return self.units[num]
        elif num <= 19:
            return self.teens[num]
        elif num in self.tens:
            return self.tens[num]
        elif num < 100:
            tens_digit = (num // 10) * 10
            units_digit = num % 10
            return f"{self.tens[tens_digit]} {self.units[units_digit]}"
        elif num == 100:
            return self.scale[100]
        elif num < 1000:
            hundreds = num // 100
            remainder = num % 100
            result = f"{self.units[hundreds]} {self.scale[100]}"
            if remainder > 0:
                result += f" {self.convert_number(remainder)}"
            return result
        elif num == 1000:
            return self.scale[1000]
        elif num < 1000000:
            thousands = num // 1000
            remainder = num % 1000
            result = f"{self.convert_number(thousands)} {self.scale[1000]}"
            if remainder > 0:
                result += f" {self.convert_number(remainder)}"
            return result
        else:
            # For very large numbers, use a simplified approach
            return str(num)  # Fallback to digits for now

    def convert_decimal(self, decimal_str: str) -> str:
        """Convert decimal numbers to Twi."""
        if '.' not in decimal_str:
            return self.convert_number(int(decimal_str))

        integer_part, decimal_part = decimal_str.split('.')
        integer_twi = self.convert_number(int(integer_part))

        # Convert decimal digits individually
        decimal_twi = ' '.join(self.units[int(d)] for d in decimal_part)

        return f"{integer_twi} ne {decimal_twi}"

    def process_all_numbers(self, text: str) -> str:
        """Process all numbers in text to Twi words."""
        def convert_match(match):
            num_str = match.group(0)

            # Handle decimals
            if '.' in num_str:
                return self.convert_decimal(num_str)

            # Handle ordinals (1st, 2nd, etc.)
            if re.search(r'\d+(st|nd|rd|th)', num_str):
                num = int(re.search(r'\d+', num_str).group())
                return self.convert_number(num, ordinal=True)

            # Regular numbers
            num = int(num_str)
            return self.convert_number(num)

        # Match decimal numbers, ordinals, and regular numbers
        pattern = r'\b\d+(?:\.\d+)?(?:st|nd|rd|th)?\b'
        text = re.sub(pattern, convert_match, text)
        return text

class TwiTextNormalizer:
    """Advanced text normalization for Twi."""

    def __init__(self):
        self.linguistic_features = TwiLinguisticFeatures()

        # Common encoding fixes
        self.encoding_fixes = {
            'è': 'ɛ', 'È': 'Ɛ', 'e`': 'ɛ', 'E`': 'Ɛ',
            'ò': 'ɔ', 'Ò': 'Ɔ', 'o`': 'ɔ', 'O`': 'Ɔ',
            'â': 'ã', 'Â': 'Ã', 'ê': 'ẽ', 'Ê': 'Ẽ',
            'î': 'ĩ', 'Î': 'Ĩ', 'ô': 'õ', 'Ô': 'Õ',
            'û': 'ũ', 'Û': 'Ũ'
        }

        # Common abbreviations
        self.abbreviations = {
            'Dr.': 'Dɔkta', 'Mr.': 'Owura', 'Mrs.': 'Awuraa',
            'Prof.': 'Profesa', 'Rev.': 'Osofoɔ',
            'vs.': 'ne', 'etc.': 'ne nea ɛkeka ho',
            '&': 'ne', '+': 'ne'
        }

        # Time and date expressions
        self.time_expressions = {
            'am': 'anɔpa', 'pm': 'anwummer',
            'AM': 'anɔpa', 'PM': 'anwummer'
        }

    def normalize_encoding(self, text: str) -> str:
        """Fix common encoding issues."""
        # Apply Unicode normalization first
        text = unicodedata.normalize('NFC', text)

        # Fix common encoding problems
        for old, new in self.encoding_fixes.items():
            text = text.replace(old, new)

        return text

    def normalize_abbreviations(self, text: str) -> str:
        """Expand common abbreviations."""
        for abbrev, expansion in self.abbreviations.items():
            # Use word boundaries to avoid partial replacements
            pattern = r'\b' + re.escape(abbrev) + r'\b'
            text = re.sub(pattern, expansion, text, flags=re.IGNORECASE)

        return text

    def normalize_punctuation(self, text: str) -> str:
        """Normalize punctuation marks."""
        # Normalize quotes
        text = re.sub(r'[\u201c\u201d]', '"', text)
        text = re.sub(r'[\u2018\u2019]', "'", text)

        # Normalize dashes
        text = re.sub(r'[–—]', '-', text)

        # Clean up multiple punctuation
        text = re.sub(r'([.!?]){2,}', r'\1', text)
        text = re.sub(r'([,;:]){2,}', r'\1', text)

        # Add spaces around punctuation if missing
        text = re.sub(r'([.!?])([A-ZƐƆ])', r'\1 \2', text)

        return text

    def normalize_whitespace(self, text: str) -> str:
        """Normalize whitespace."""
        # Replace multiple spaces with single space
        text = re.sub(r'\s+', ' ', text)

        # Remove leading/trailing whitespace
        text = text.strip()

        # Fix spacing around punctuation
        text = re.sub(r'\s+([.!?,:;])', r'\1', text)
        text = re.sub(r'([.!?])\s*([A-ZƐƆ])', r'\1 \2', text)

        return text

    def normalize_case(self, text: str, preserve_proper_nouns: bool = True) -> str:
        """Normalize text case while preserving proper nouns."""
        if not preserve_proper_nouns:
            return text.lower()

        # Split into sentences
        sentences = re.split(r'([.!?]+)', text)
        normalized_sentences = []

        for i, sentence in enumerate(sentences):
            if i % 2 == 0:  # Actual sentence content
                # Capitalize first word of sentence
                words = sentence.strip().split()
                if words:
                    words[0] = words[0].capitalize()

                    # Keep proper nouns capitalized (heuristic approach)
                    for j, word in enumerate(words[1:], 1):
                        # If word starts with capital and is not a common Twi word
                        if (word and word[0].isupper() and
                            word.lower() not in self.linguistic_features.common_words):
                            continue  # Keep capitalized
                        else:
                            words[j] = word.lower()

                normalized_sentences.append(' '.join(words))
            else:
                normalized_sentences.append(sentence)  # Punctuation

        return ''.join(normalized_sentences)

class TwiTextQualityAssessor:
    """Assess the quality of Twi text."""

    def __init__(self):
        self.linguistic_features = TwiLinguisticFeatures()

        # Quality indicators
        self.positive_indicators = {
            'twi_characters': 2.0,      # Contains ɛ, ɔ
            'common_words': 1.5,        # Contains common Twi words
            'proper_length': 1.0,       # Appropriate length
            'punctuation': 0.5,         # Has proper punctuation
            'capitalization': 0.5       # Proper capitalization
        }

        self.negative_indicators = {
            'excessive_english': -2.0,   # Too much English
            'encoding_issues': -1.5,     # Encoding problems
            'excessive_punctuation': -1.0,  # Too much punctuation
            'too_short': -2.0,          # Very short
            'too_long': -1.0,           # Excessively long
            'no_vowels': -3.0           # Missing vowels (corruption)
        }

    def assess_quality(self, text: str) -> Dict[str, float]:
        """Assess text quality and return detailed scores."""
        if not text or len(text.strip()) == 0:
            return {
                'overall_score': 0.0,
                'length_score': 0.0,
                'linguistic_score': 0.0,
                'encoding_score': 0.0,
                'structure_score': 0.0,
                'issues': ['empty_text']
            }

        text = text.strip()
        issues = []

        # Length assessment
        length_score = self._assess_length(text, issues)

        # Linguistic content assessment
        linguistic_score = self._assess_linguistic_content(text, issues)

        # Encoding quality assessment
        encoding_score = self._assess_encoding_quality(text, issues)

        # Structure assessment
        structure_score = self._assess_structure(text, issues)

        # Overall score (weighted average)
        overall_score = (
            length_score * 0.2 +
            linguistic_score * 0.4 +
            encoding_score * 0.2 +
            structure_score * 0.2
        )

        return {
            'overall_score': max(0.0, min(10.0, overall_score)),
            'length_score': length_score,
            'linguistic_score': linguistic_score,
            'encoding_score': encoding_score,
            'structure_score': structure_score,
            'issues': issues
        }

    def _assess_length(self, text: str, issues: List[str]) -> float:
        """Assess text length appropriateness."""
        char_count = len(text)
        word_count = len(text.split())

        if char_count < 5:
            issues.append('too_short')
            return 1.0
        elif char_count > 500:
            issues.append('too_long')
            return 6.0
        elif 10 <= char_count <= 200:
            return 8.0
        elif 5 <= char_count < 10 or 200 < char_count <= 500:
            return 6.0
        else:
            return 7.0

    def _assess_linguistic_content(self, text: str, issues: List[str]) -> float:
        """Assess linguistic content quality."""
        text_lower = text.lower()
        score = 5.0

        # Check for Twi-specific characters
        if any(char in text for char in self.linguistic_features.twi_special_chars):
            score += self.positive_indicators['twi_characters']

        # Check for common Twi words
        words = text_lower.split()
        twi_word_count = sum(1 for word in words if word in self.linguistic_features.common_words)
        if twi_word_count > 0:
            score += self.positive_indicators['common_words'] * min(twi_word_count / len(words), 1.0)

        # Check for excessive English
        english_indicators = ['the', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for']
        english_count = sum(1 for word in words if word in english_indicators)
        if english_count / len(words) > 0.3:  # More than 30% English
            issues.append('excessive_english')
            score += self.negative_indicators['excessive_english']

        # Check for vowels (corruption indicator)
        if not any(char in text_lower for char in 'aeiouɛɔ'):
            issues.append('no_vowels')
            score += self.negative_indicators['no_vowels']

        return max(1.0, min(10.0, score))

    def _assess_encoding_quality(self, text: str, issues: List[str]) -> float:
        """Assess encoding quality."""
        score = 8.0

        # Check for common encoding issues
        encoding_issues = ['Ã', 'â', 'Â', 'ê', 'Ê', 'î', 'Î', 'ô', 'Ô', 'û', 'Û']
        if any(issue in text for issue in encoding_issues):
            issues.append('encoding_issues')
            score += self.negative_indicators['encoding_issues']

        # Check for control characters
        if any(ord(char) < 32 and char not in '\n\r\t' for char in text):
            issues.append('control_characters')
            score -= 2.0

        return max(1.0, min(10.0, score))

    def _assess_structure(self, text: str, issues: List[str]) -> float:
        """Assess text structure quality."""
        score = 6.0

        # Check punctuation
        if re.search(r'[.!?]', text):
            score += self.positive_indicators['punctuation']

        # Check capitalization
        if re.search(r'^[A-ZƐƆ]', text):
            score += self.positive_indicators['capitalization']

        # Check excessive punctuation
        punctuation_density = len(re.findall(r'[.!?,;:]', text)) / len(text)
        if punctuation_density > 0.1:  # More than 10% punctuation
            issues.append('excessive_punctuation')
            score += self.negative_indicators['excessive_punctuation']

        return max(1.0, min(10.0, score))

class EnhancedTwiTextProcessor:
    """Complete high-quality Twi text processing pipeline."""

    def __init__(self, dialect: str = 'asante', preserve_formatting: bool = True):
        self.dialect = dialect
        self.preserve_formatting = preserve_formatting

        # Initialize components
        self.number_processor = TwiNumberProcessor()
        self.normalizer = TwiTextNormalizer()
        self.quality_assessor = TwiTextQualityAssessor()
        self.linguistic_features = TwiLinguisticFeatures()

        logger.info(f"Enhanced Twi text processor initialized for {dialect} dialect")

    def process_text(self, text: str, expand_numbers: bool = True,
                    assess_quality: bool = True, normalize_case: bool = True) -> Dict:
        """Complete text processing pipeline with comprehensive analysis."""
        if not text:
            return self._empty_result()

        original_text = text
        processing_log = []

        # Step 1: Initial cleanup
        text = text.strip()
        if not text:
            return self._empty_result(original_text)

        # Step 2: Encoding normalization
        text = self.normalizer.normalize_encoding(text)
        processing_log.append('encoding_normalized')

        # Step 3: Abbreviation expansion
        text = self.normalizer.normalize_abbreviations(text)
        processing_log.append('abbreviations_expanded')

        # Step 4: Number processing
        if expand_numbers:
            text = self.number_processor.process_all_numbers(text)
            processing_log.append('numbers_converted')

        # Step 5: Punctuation normalization
        text = self.normalizer.normalize_punctuation(text)
        processing_log.append('punctuation_normalized')

        # Step 6: Case normalization
        if normalize_case:
            text = self.normalizer.normalize_case(text, preserve_proper_nouns=True)
            processing_log.append('case_normalized')

        # Step 7: Whitespace cleanup
        text = self.normalizer.normalize_whitespace(text)
        processing_log.append('whitespace_normalized')

        # Step 8: Quality assessment
        quality_analysis = None
        if assess_quality:
            quality_analysis = self.quality_assessor.assess_quality(text)
            processing_log.append('quality_assessed')

        # Step 9: Linguistic analysis
        linguistic_analysis = self._analyze_linguistic_features(text)
        processing_log.append('linguistic_analyzed')

        # Compile results
        result = {
            'original': original_text,
            'processed': text,
            'dialect': self.dialect,
            'char_count': len(text),
            'word_count': len(text.split()),
            'has_twi_chars': linguistic_analysis['has_twi_chars'],
            'twi_char_count': linguistic_analysis['twi_char_count'],
            'common_twi_words': linguistic_analysis['common_twi_words'],
            'estimated_twi_content': linguistic_analysis['estimated_twi_content'],
            'processing_log': processing_log,
            'changes_made': original_text.strip() != text
        }

        if quality_analysis:
            result['quality'] = quality_analysis

        return result

    def _analyze_linguistic_features(self, text: str) -> Dict:
        """Analyze linguistic features of the text."""
        text_lower = text.lower()
        words = text_lower.split()

        # Count Twi-specific characters
        twi_chars = [char for char in text if char in self.linguistic_features.twi_special_chars]

        # Count common Twi words
        common_words = [word for word in words if word in self.linguistic_features.common_words]

        # Estimate Twi content percentage
        twi_indicators = len(twi_chars) + len(common_words) * 2
        total_indicators = len(text) + len(words)
        estimated_twi_content = min(1.0, twi_indicators / total_indicators) if total_indicators > 0 else 0.0

        return {
            'has_twi_chars': len(twi_chars) > 0,
            'twi_char_count': len(twi_chars),
            'common_twi_words': len(common_words),
            'estimated_twi_content': estimated_twi_content,
            'unique_twi_chars': list(set(twi_chars)),
            'found_common_words': common_words
        }

    def _empty_result(self, original: str = '') -> Dict:
        """Return result structure for empty/invalid text."""
        return {
            'original': original,
            'processed': '',
            'dialect': self.dialect,
            'char_count': 0,
            'word_count': 0,
            'has_twi_chars': False,
            'twi_char_count': 0,
            'common_twi_words': 0,
            'estimated_twi_content': 0.0,
            'processing_log': ['empty_input'],
            'changes_made': False,
            'quality': {
                'overall_score': 0.0,
                'length_score': 0.0,
                'linguistic_score': 0.0,
                'encoding_score': 0.0,
                'structure_score': 0.0,
                'issues': ['empty_text']
            }
        }

    def batch_process_texts(self, texts: List[str], **kwargs) -> List[Dict]:
        """Process multiple texts efficiently."""
        results = []

        for i, text in enumerate(texts):
            try:
                result = self.process_text(text, **kwargs)
                results.append(result)

                # Log progress for large batches
                if i > 0 and i % 1000 == 0:
                    logger.info(f"Processed {i}/{len(texts)} texts")

            except Exception as e:
                logger.error(f"Failed to process text {i}: {e}")
                results.append(self._empty_result(text))

        return results

    def get_processing_statistics(self, results: List[Dict]) -> Dict:
        """Get processing statistics from batch results."""
        if not results:
            return {'total': 0, 'processed': 0, 'success_rate': 0.0}

        total = len(results)
        processed = sum(1 for r in results if r['processed'])
        high_quality = sum(1 for r in results if r.get('quality', {}).get('overall_score', 0) >= 7.0)

        return {
            'total_texts': total,
            'successfully_processed': processed,
            'high_quality_texts': high_quality,
            'success_rate': processed / total if total > 0 else 0.0,
            'quality_rate': high_quality / total if total > 0 else 0.0,
            'average_length': sum(r['char_count'] for r in results) / total if total > 0 else 0.0,
            'twi_content_rate': sum(1 for r in results if r['has_twi_chars']) / total if total > 0 else 0.0
        }

# Test function for validation
def test_enhanced_processor():
    """Test the enhanced text processor."""
    processor = EnhancedTwiTextProcessor()

    test_texts = [
        "Wo ho te sɛn? Me ho yɛ.",
        "Me ma wo akye 5 berɛ wɔ ɔkyena anɔpa no.",
        "Dr. Kwame ne Prof. Ama baa sukuu no mu",
        "Ɛyɛ 23.5 degrees wɔ Accra mu nnɛ",
        "Me pɛ sɛ menya 1st prize wɔ competition yi mu"
    ]

    print("🧪 Testing Enhanced Twi Text Processor")
    print("=" * 50)

    for i, text in enumerate(test_texts, 1):
        result = processor.process_text(text)
        print(f"\nTest {i}:")
        print(f"  Original: {result['original']}")
        print(f"  Processed: {result['processed']}")
        print(f"  Quality: {result['quality']['overall_score']:.1f}/10")
        print(f"  Changes: {result['changes_made']}")
        print(f"  Twi content: {result['estimated_twi_content']:.1%}")

    print("\n✅ Testing complete!")

if __name__ == "__main__":
    test_enhanced_processor()

print("[SUCCESS] Enhanced Twi Text Processor loaded successfully!")

Overwriting enhanced_text_processor.py


In [10]:
# Import the enhanced processing system
from enhanced_resilient_data_processor import (
    EnhancedMultiRateConfig,
    EnhancedDatasetValidator,
    ProgressiveResilientDataLoader,
    MemoryManager
)
from enhanced_text_processor import EnhancedTwiTextProcessor
from pathlib import Path
import pandas as pd
import numpy as np

print("🚀 Enhanced resilient processing system imported successfully!")
print("🎯 Features: Progressive + Validation + Error Handling + Memory Management")

[SUCCESS] Enhanced Twi Text Processor loaded successfully!
🚀 Enhanced resilient processing system imported successfully!
🎯 Features: Progressive + Validation + Error Handling + Memory Management


In [11]:
# Initialize enhanced configuration
BASE_PATH = Path('/content/drive/MyDrive/voice-lab-data')

config = EnhancedMultiRateConfig(
    base_path=BASE_PATH,
    output_path=BASE_PATH / 'processed_resilient',
    batch_size=300,  # Smaller batches for memory efficiency
    max_samples_per_dataset=3000,  # Reasonable limit per dataset
    max_arrow_files_per_dataset=8,  # Prevent memory overload
    memory_check_interval=50  # More frequent memory checks
)

# Create output directories
config.output_path.mkdir(parents=True, exist_ok=True)

print(f"🔧 Enhanced configuration initialized:")
print(f"   Base path: {config.base_path}")
print(f"   Output path: {config.output_path}")
print(f"   Sample rates: {config.sample_rates}")
print(f"   Batch size: {config.batch_size}")
print(f"   Max samples per dataset: {config.max_samples_per_dataset}")
print(f"   Datasets configured: {len(config.datasets_to_scan)}")

# Initialize memory manager
memory_manager = MemoryManager(max_memory_percent=80)
initial_memory = memory_manager.check_memory()
print(f"\n💾 Initial memory status: {initial_memory['used_mb']:.1f}MB used ({initial_memory['percent_used']:.1f}%)")

🔧 Enhanced configuration initialized:
   Base path: /content/drive/MyDrive/voice-lab-data
   Output path: /content/drive/MyDrive/voice-lab-data/processed_resilient
   Sample rates: {'asr_16k': 16000, 'tts_22k': 22050, 'tts_24k': 24000}
   Batch size: 300
   Max samples per dataset: 3000
   Datasets configured: 9

💾 Initial memory status: 179.0MB used (7.9%)


In [12]:
# Step 1: Comprehensive Dataset Validation
print("🔍 STEP 1: Comprehensive Dataset Validation")
print("=" * 50)

validator = EnhancedDatasetValidator(config.base_path)
valid_configs, validation_report = validator.validate_all_datasets(config.datasets_to_scan)

print(f"\n📊 Validation Results:")
print(f"   Valid datasets: {len(valid_configs)}/{len(config.datasets_to_scan)}")
print(f"   Estimated total samples: {validation_report['estimated_total_samples']:,}")
print(f"   Success rate: {validation_report['valid_datasets']/validation_report['total_datasets']*100:.1f}%")

if valid_configs:
    print("\n✅ Valid datasets found:")
    for config_item in valid_configs:
        details = validation_report['details'][config_item['name']]
        print(f"   📂 {config_item['name']}: {details['estimated_samples']:,} samples ({details['confidence']} confidence)")
else:
    print("\n⚠️ No valid datasets found - will use dummy data for testing")

# Update config to use only valid datasets
config.datasets_to_scan = valid_configs

print("\n✅ Step 1 Complete: Dataset validation finished")

🔍 STEP 1: Comprehensive Dataset Validation


Validating datasets:   0%|          | 0/9 [00:00<?, ?it/s]


📊 Validation Results:
   Valid datasets: 1/9
   Estimated total samples: 2,000
   Success rate: 11.1%

✅ Valid datasets found:
   📂 bibletts: 2,000 samples (high confidence)

✅ Step 1 Complete: Dataset validation finished


In [ ]:
# Step 2: Progressive Resilient Data Loading
print("\n🚀 STEP 2: Progressive Resilient Data Loading")
print("=" * 50)

# Initialize progressive loader
progressive_loader = ProgressiveResilientDataLoader(config)

# Check memory before starting
memory_status = memory_manager.check_memory()
print(f"💾 Starting memory: {memory_status['used_mb']:.1f}MB ({memory_status['percent_used']:.1f}%)")

# Load all datasets progressively
raw_dataset = progressive_loader.load_all_datasets_progressive(valid_configs)

print(f"\n📊 Raw Dataset Summary:")
print(f"   Total samples: {len(raw_dataset):,}")

if not raw_dataset.empty:
    print(f"   Unique speakers: {raw_dataset['speaker_id'].nunique():,}")
    print(f"   Datasets: {list(raw_dataset['dataset'].unique())}")
    print(f"   Languages: {list(raw_dataset['language'].unique())}")
    print(f"   Total duration: {raw_dataset['duration'].sum()/3600:.1f} hours")
    print(f"   Quality samples: {raw_dataset['passes_quality_filter'].sum():,} ({raw_dataset['passes_quality_filter'].mean()*100:.1f}%)")

    # Show sample
    print(f"\n📝 Sample Records:")
    display(raw_dataset[['dataset', 'speaker_id', 'text', 'duration', 'quality_score', 'passes_quality_filter']].head())
else:
    print("   ⚠️ No data loaded - using dummy dataset")

# Check failed datasets
if progressive_loader.failed_datasets:
    print(f"\n⚠️ Failed datasets: {progressive_loader.failed_datasets}")
else:
    print(f"\n✅ All {len(valid_configs)} datasets loaded successfully!")

# Memory status after loading
final_memory = memory_manager.check_memory()
print(f"\n💾 Final memory: {final_memory['used_mb']:.1f}MB ({final_memory['percent_used']:.1f}%)")
print(f"💾 Memory increase: {final_memory['used_mb'] - memory_status['used_mb']:.1f}MB")

print("\n✅ Step 2 Complete: Progressive data loading finished")


🚀 STEP 2: Progressive Resilient Data Loading
💾 Starting memory: 187.5MB (17.5%)


Loading datasets:   0%|          | 0/1 [00:00<?, ?it/s]

Processing bibletts:   0%|          | 0/2000 [00:00<?, ?it/s]

In [ ]:
# Step 3: Multi-Rate Audio Processing with Memory Management
print("\n🎵 STEP 3: Multi-Rate Audio Processing")
print("=" * 50)

# Use only quality samples for processing to maximize efficiency
quality_dataset = raw_dataset[raw_dataset['passes_quality_filter']].copy()
print(f"🔍 Processing {len(quality_dataset):,} quality samples (filtered from {len(raw_dataset):,} total)")

# Import audio processor from original system (enhanced version)
from enhanced_resilient_data_processor import MultiRateAudioProcessor

class MemoryEfficientAudioProcessor(MultiRateAudioProcessor):
    """Memory-efficient version of audio processor."""

    def __init__(self, config, memory_manager):
        super().__init__(config)
        self.memory_manager = memory_manager

    def process_for_all_rates(self, df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
        """Process with memory management."""
        print("🎵 Processing audio with memory management...")

        processed_datasets = {}

        for rate_category, sample_rate in self.config.sample_rates.items():
            print(f"\n--- Processing {rate_category} ({sample_rate}Hz) ---")

            # Check memory before each rate
            memory = self.memory_manager.check_memory()
            if memory['is_critical']:
                print("⚠️ Memory critical - forcing cleanup")
                self.memory_manager.force_gc()

            # Create output directories
            audio_dir = self.config.output_path / 'processed_audio' / rate_category
            audio_dir.mkdir(parents=True, exist_ok=True)

            # Process in smaller batches
            batch_size = min(200, len(df))  # Smaller batches
            processed_samples = []

            for start_idx in tqdm(range(0, len(df), batch_size), desc=f"Processing {rate_category} batches"):
                end_idx = min(start_idx + batch_size, len(df))
                batch_df = df.iloc[start_idx:end_idx]

                batch_processed = self._process_batch_safe(batch_df, rate_category, sample_rate, audio_dir)
                processed_samples.extend(batch_processed)

                # Cleanup after each batch
                if start_idx % (batch_size * 5) == 0:  # Every 5 batches
                    self.memory_manager.force_gc()

            if processed_samples:
                processed_df = pd.DataFrame(processed_samples)
                processed_datasets[rate_category] = processed_df

                print(f"   ✅ Processed {len(processed_df)} samples for {rate_category}")

                # Save manifest
                manifest_path = self.config.output_path / 'manifests' / f'{rate_category}_dataset.csv'
                manifest_path.parent.mkdir(parents=True, exist_ok=True)
                processed_df.to_csv(manifest_path, index=False)

        return processed_datasets

    def _process_batch_safe(self, batch_df, rate_category, sample_rate, audio_dir):
        """Process batch with error handling."""
        processed_samples = []

        for _, row in batch_df.iterrows():
            try:
                # Load and process audio
                if not Path(row['audio_path']).exists():
                    continue

                audio, sr = librosa.load(row['audio_path'], sr=sample_rate, mono=True)

                # Apply normalization
                audio = self._normalize_audio(audio)

                # Create output filename
                output_filename = f"{row['dataset']}_{row['speaker_id']}_{Path(row['audio_path']).stem}.wav"
                output_path = audio_dir / output_filename

                # Save processed audio
                sf.write(output_path, audio, sample_rate)

                # Create sample record
                sample_data = row.to_dict()
                sample_data.update({
                    'processed_audio_path': str(output_path),
                    'processed_sample_rate': sample_rate,
                    'processed_duration': len(audio) / sample_rate,
                    'rate_category': rate_category
                })

                processed_samples.append(sample_data)

            except Exception as e:
                logger.debug(f"Failed to process {row.get('audio_path', 'unknown')}: {e}")
                continue

        return processed_samples

# Initialize memory-efficient processor
audio_processor = MemoryEfficientAudioProcessor(config, memory_manager)
processed_datasets = audio_processor.process_for_all_rates(quality_dataset)

print(f"\n✅ Processing Complete!")
print(f"   Processed datasets: {list(processed_datasets.keys())}")

for rate_category, df in processed_datasets.items():
    sample_rate = config.sample_rates[rate_category]
    total_duration = df['processed_duration'].sum() / 3600
    print(f"   {rate_category} ({sample_rate}Hz): {len(df):,} samples, {total_duration:.1f} hours")

print("\n✅ Step 3 Complete: Multi-rate audio processing finished")

In [ ]:
# Step 4: Enhanced Model-Specific Manifest Generation
print("\n📋 STEP 4: Enhanced Model-Specific Manifest Generation")
print("=" * 50)

from enhanced_resilient_data_processor import ModelSpecificManifestGenerator

class EnhancedManifestGenerator(ModelSpecificManifestGenerator):
    """Enhanced manifest generator with quality filtering."""

    def generate_all_manifests(self, processed_datasets: Dict[str, pd.DataFrame]):
        """Generate manifests with quality-based splits."""
        print("📋 Generating enhanced model-specific manifests...")

        manifest_dir = self.config.output_path / 'model_manifests'
        manifest_dir.mkdir(parents=True, exist_ok=True)

        for rate_category, models in self.config.model_categories.items():
            if rate_category not in processed_datasets:
                print(f"   ⚠️ No processed data for {rate_category}")
                continue

            df = processed_datasets[rate_category]

            # Quality-based filtering for manifests
            high_quality = df[df['quality_score'] >= 7.0] if 'quality_score' in df.columns else df
            medium_quality = df[df['quality_score'] >= 5.0] if 'quality_score' in df.columns else df

            print(f"\n   📊 {rate_category} Quality Distribution:")
            print(f"      Total samples: {len(df):,}")
            print(f"      High quality (≥7.0): {len(high_quality):,}")
            print(f"      Medium quality (≥5.0): {len(medium_quality):,}")

            for model_name in models:
                print(f"   📝 Creating manifests for {model_name}...")

                # Use high quality for primary training, medium for augmentation
                self._create_enhanced_manifest(high_quality, medium_quality, model_name, manifest_dir, rate_category)

    def _create_enhanced_manifest(self, high_quality_df, medium_quality_df, model_name, manifest_dir, rate_category):
        """Create enhanced manifests with quality tiers."""

        # Primary training set (high quality)
        if len(high_quality_df) > 0:
            primary_train = high_quality_df.sample(frac=0.9, random_state=42)
            primary_val = high_quality_df.drop(primary_train.index)
        else:
            primary_train = pd.DataFrame()
            primary_val = pd.DataFrame()

        # Extended training set (medium quality for additional data)
        extended_samples = medium_quality_df[~medium_quality_df.index.isin(high_quality_df.index)]

        if 'whisper' in model_name or 'wav2vec2' in model_name or 'mms' in model_name or 'moshi' in model_name:
            self._save_asr_manifests(primary_train, primary_val, extended_samples, model_name, manifest_dir)
        else:
            self._save_tts_manifests(primary_train, primary_val, extended_samples, model_name, manifest_dir)

    def _save_asr_manifests(self, train_df, val_df, extended_df, model_name, manifest_dir):
        """Save ASR manifests with quality tiers."""
        # Primary high-quality manifests
        if len(train_df) > 0:
            train_data = self._prepare_asr_data(train_df)
            pd.DataFrame(train_data).to_csv(manifest_dir / f'{model_name}_train_hq.csv', index=False)

        if len(val_df) > 0:
            val_data = self._prepare_asr_data(val_df)
            pd.DataFrame(val_data).to_csv(manifest_dir / f'{model_name}_val_hq.csv', index=False)

        # Extended training set
        if len(extended_df) > 0:
            extended_data = self._prepare_asr_data(extended_df)
            pd.DataFrame(extended_data).to_csv(manifest_dir / f'{model_name}_train_extended.csv', index=False)

        # Combined manifests (for compatibility)
        all_train = pd.concat([train_df, extended_df]) if len(extended_df) > 0 else train_df
        if len(all_train) > 0:
            combined_data = self._prepare_asr_data(all_train)
            pd.DataFrame(combined_data).to_csv(manifest_dir / f'{model_name}_train.csv', index=False)

        if len(val_df) > 0:
            val_data = self._prepare_asr_data(val_df)
            pd.DataFrame(val_data).to_csv(manifest_dir / f'{model_name}_val.csv', index=False)

    def _save_tts_manifests(self, train_df, val_df, extended_df, model_name, manifest_dir):
        """Save TTS manifests with quality tiers."""
        # Similar structure for TTS models
        if len(train_df) > 0:
            if 'coqui' in model_name:
                train_lines = self._prepare_coqui_data(train_df)
                with open(manifest_dir / f'{model_name}_train_hq.txt', 'w', encoding='utf-8') as f:
                    f.write('\n'.join(train_lines))
            else:
                train_data = self._prepare_tts_data(train_df)
                pd.DataFrame(train_data).to_csv(manifest_dir / f'{model_name}_train_hq.csv', index=False)

        # Combined manifests
        all_train = pd.concat([train_df, extended_df]) if len(extended_df) > 0 else train_df
        if len(all_train) > 0:
            if 'coqui' in model_name:
                train_lines = self._prepare_coqui_data(all_train)
                val_lines = self._prepare_coqui_data(val_df) if len(val_df) > 0 else []

                with open(manifest_dir / f'{model_name}_train.txt', 'w', encoding='utf-8') as f:
                    f.write('\n'.join(train_lines))

                if val_lines:
                    with open(manifest_dir / f'{model_name}_val.txt', 'w', encoding='utf-8') as f:
                        f.write('\n'.join(val_lines))
            else:
                train_data = self._prepare_tts_data(all_train)
                val_data = self._prepare_tts_data(val_df) if len(val_df) > 0 else []

                pd.DataFrame(train_data).to_csv(manifest_dir / f'{model_name}_train.csv', index=False)

                if val_data:
                    pd.DataFrame(val_data).to_csv(manifest_dir / f'{model_name}_val.csv', index=False)

    def _prepare_asr_data(self, df):
        """Prepare ASR training data."""
        return [
            {
                'audio_path': row['processed_audio_path'],
                'transcript': row.get('processed_text', row['text']),
                'duration': row['processed_duration'],
                'speaker_id': row['speaker_id'],
                'language': row['language'],
                'dataset': row['dataset'],
                'quality_score': row.get('quality_score', 5.0)
            }
            for _, row in df.iterrows()
        ]

    def _prepare_tts_data(self, df):
        """Prepare TTS training data."""
        return [
            {
                'audio_path': row['processed_audio_path'],
                'text': row.get('processed_text', row['text']),
                'speaker_id': row['speaker_id'],
                'language': row['language'],
                'duration': row['processed_duration'],
                'quality_score': row.get('quality_score', 5.0)
            }
            for _, row in df.iterrows()
        ]

    def _prepare_coqui_data(self, df):
        """Prepare Coqui TTS format data."""
        return [
            f"wavs/{Path(row['processed_audio_path']).name}|{row.get('processed_text', row['text'])}|{row['speaker_id']}"
            for _, row in df.iterrows()
        ]

# Generate enhanced manifests
enhanced_generator = EnhancedManifestGenerator(config)
enhanced_generator.generate_all_manifests(processed_datasets)

print("\n✅ Enhanced manifests generated!")

# List generated manifests
manifest_dir = config.output_path / 'model_manifests'
if manifest_dir.exists():
    manifests = sorted(list(manifest_dir.glob('*.csv')) + list(manifest_dir.glob('*.txt')))
    print(f"\n📄 Generated manifests ({len(manifests)}):")
    for manifest in manifests:
        size_kb = manifest.stat().st_size / 1024
        print(f"   {manifest.name} ({size_kb:.1f}KB)")

print("\n✅ Step 4 Complete: Enhanced manifest generation finished")

In [ ]:
# Step 5: Comprehensive Quality Assessment & Statistics
print("\n📊 STEP 5: Comprehensive Quality Assessment")
print("=" * 50)

from enhanced_resilient_data_processor import DatasetStatistics

class EnhancedStatistics(DatasetStatistics):
    """Enhanced statistics with quality analysis."""

    def generate_comprehensive_report(self, raw_df, processed_datasets, validation_report):
        """Generate comprehensive analysis report."""

        report = {
            'processing_summary': {
                'total_datasets_attempted': validation_report['total_datasets'],
                'valid_datasets': validation_report['valid_datasets'],
                'datasets_loaded': len(processed_datasets),
                'validation_success_rate': validation_report['valid_datasets'] / validation_report['total_datasets'],
            },
            'data_quality': {
                'raw_samples': len(raw_df),
                'quality_samples': raw_df['passes_quality_filter'].sum() if 'passes_quality_filter' in raw_df.columns else 0,
                'quality_retention_rate': raw_df['passes_quality_filter'].mean() if 'passes_quality_filter' in raw_df.columns else 0,
                'average_quality_score': raw_df['quality_score'].mean() if 'quality_score' in raw_df.columns else 0,
            },
            'audio_statistics': self._analyze_audio_stats(raw_df),
            'text_statistics': self._analyze_text_stats(raw_df),
            'processing_efficiency': self._analyze_processing_efficiency(raw_df, processed_datasets),
            'model_readiness': self._analyze_model_readiness(processed_datasets)
        }

        return report

    def _analyze_audio_stats(self, df):
        """Analyze audio statistics."""
        if df.empty or 'duration' not in df.columns:
            return {}

        return {
            'total_hours': df['duration'].sum() / 3600,
            'average_duration': df['duration'].mean(),
            'median_duration': df['duration'].median(),
            'duration_std': df['duration'].std(),
            'short_samples': (df['duration'] < 2.0).sum(),
            'medium_samples': ((df['duration'] >= 2.0) & (df['duration'] <= 10.0)).sum(),
            'long_samples': (df['duration'] > 10.0).sum(),
            'optimal_duration_rate': ((df['duration'] >= 2.0) & (df['duration'] <= 15.0)).mean()
        }

    def _analyze_text_stats(self, df):
        """Analyze text statistics."""
        if df.empty:
            return {}

        stats = {
            'average_text_length': df['text_length'].mean() if 'text_length' in df.columns else 0,
            'average_word_count': df['word_count'].mean() if 'word_count' in df.columns else 0,
            'twi_content_rate': df['has_twi_chars'].mean() if 'has_twi_chars' in df.columns else 0,
        }

        # Dialect distribution
        if 'dialect' in df.columns:
            stats['dialect_distribution'] = df['dialect'].value_counts().to_dict()

        return stats

    def _analyze_processing_efficiency(self, raw_df, processed_datasets):
        """Analyze processing efficiency."""
        total_processed = sum(len(df) for df in processed_datasets.values())

        return {
            'raw_to_processed_rate': total_processed / len(raw_df) if len(raw_df) > 0 else 0,
            'samples_per_rate_category': {cat: len(df) for cat, df in processed_datasets.items()},
            'processing_success_rate': total_processed / (len(raw_df) * len(processed_datasets)) if len(raw_df) > 0 else 0
        }

    def _analyze_model_readiness(self, processed_datasets):
        """Analyze readiness for model training."""
        readiness = {}

        for rate_category, df in processed_datasets.items():
            models = config.model_categories.get(rate_category, [])

            total_hours = df['processed_duration'].sum() / 3600 if 'processed_duration' in df.columns else 0
            unique_speakers = df['speaker_id'].nunique() if 'speaker_id' in df.columns else 0

            # Assess readiness based on data volume and speaker diversity
            readiness_score = min(1.0, (
                (total_hours / 10.0) * 0.4 +  # Target: 10+ hours
                (unique_speakers / 50.0) * 0.3 +  # Target: 50+ speakers
                (len(df) / 1000.0) * 0.3  # Target: 1000+ samples
            ))

            readiness[rate_category] = {
                'models': models,
                'total_hours': total_hours,
                'unique_speakers': unique_speakers,
                'total_samples': len(df),
                'readiness_score': readiness_score,
                'ready_for_training': readiness_score >= 0.5
            }

        return readiness

# Generate comprehensive report
enhanced_stats = EnhancedStatistics(config)
comprehensive_report = enhanced_stats.generate_comprehensive_report(
    raw_dataset, processed_datasets, validation_report
)

# Display key findings
print("📈 COMPREHENSIVE ANALYSIS RESULTS")
print("=" * 50)

print(f"\n🔍 Processing Summary:")
ps = comprehensive_report['processing_summary']
print(f"   Datasets attempted: {ps['total_datasets_attempted']}")
print(f"   Valid datasets: {ps['valid_datasets']}")
print(f"   Successfully loaded: {ps['datasets_loaded']}")
print(f"   Overall success rate: {ps['validation_success_rate']*100:.1f}%")

print(f"\n📊 Data Quality:")
dq = comprehensive_report['data_quality']
print(f"   Raw samples: {dq['raw_samples']:,}")
print(f"   Quality samples: {dq['quality_samples']:,}")
print(f"   Quality retention: {dq['quality_retention_rate']*100:.1f}%")
print(f"   Average quality score: {dq['average_quality_score']:.1f}/10")

print(f"\n🎵 Audio Statistics:")
aus = comprehensive_report['audio_statistics']
print(f"   Total duration: {aus.get('total_hours', 0):.1f} hours")
print(f"   Average duration: {aus.get('average_duration', 0):.1f}s")
print(f"   Optimal duration rate: {aus.get('optimal_duration_rate', 0)*100:.1f}%")

print(f"\n📝 Text Statistics:")
ts = comprehensive_report['text_statistics']
print(f"   Average text length: {ts.get('average_text_length', 0):.0f} chars")
print(f"   Average word count: {ts.get('average_word_count', 0):.1f} words")
print(f"   Twi content rate: {ts.get('twi_content_rate', 0)*100:.1f}%")

print(f"\n🤖 Model Readiness:")
mr = comprehensive_report['model_readiness']
for category, info in mr.items():
    status = "✅ READY" if info['ready_for_training'] else "⚠️ NEEDS MORE DATA"
    print(f"   {category}: {status} ({info['readiness_score']*100:.0f}% ready)")
    print(f"     - {info['total_hours']:.1f}h, {info['unique_speakers']} speakers, {info['total_samples']} samples")

# Save comprehensive report
report_path = config.output_path / 'comprehensive_analysis_report.json'
with open(report_path, 'w') as f:
    json.dump(comprehensive_report, f, indent=2, default=str)

print(f"\n📄 Comprehensive report saved: {report_path}")
print("\n✅ Step 5 Complete: Quality assessment and statistics finished")

In [ ]:
# Step 6: Final Validation & Next Steps
print("\n🏁 STEP 6: Final Validation & Next Steps")
print("=" * 50)

# Final memory check
final_memory = memory_manager.check_memory()
print(f"💾 Final memory usage: {final_memory['used_mb']:.1f}MB ({final_memory['percent_used']:.1f}%)")

# Validation checks
validation_results = {}
all_passed = True

for rate_category, df in processed_datasets.items():
    print(f"\n🔍 Validating {rate_category}...")

    # Check audio files exist
    if 'processed_audio_path' in df.columns:
        existing_files = df['processed_audio_path'].apply(lambda x: Path(x).exists()).sum()
        file_existence_rate = existing_files / len(df)
    else:
        existing_files = 0
        file_existence_rate = 0

    # Check manifest files
    manifest_dir = config.output_path / 'model_manifests'
    models = config.model_categories[rate_category]
    manifest_files = []

    for model in models:
        train_file = manifest_dir / f'{model}_train.csv'
        val_file = manifest_dir / f'{model}_val.csv'
        train_txt = manifest_dir / f'{model}_train.txt'
        val_txt = manifest_dir / f'{model}_val.txt'

        if (train_file.exists() and val_file.exists()) or (train_txt.exists() and val_txt.exists()):
            manifest_files.append(model)

    validation_results[rate_category] = {
        'samples': len(df),
        'audio_files_exist': existing_files,
        'file_existence_rate': file_existence_rate,
        'manifest_models': len(manifest_files),
        'expected_models': len(models),
        'manifest_success_rate': len(manifest_files) / len(models) if models else 0
    }

    # Display results
    print(f"   📊 Samples: {len(df):,}")
    print(f"   📁 Audio files: {existing_files}/{len(df)} ({file_existence_rate:.1%})")
    print(f"   📋 Manifests: {len(manifest_files)}/{len(models)} models")

    # Check if passed
    passed = file_existence_rate >= 0.95 and len(manifest_files) == len(models)
    status = "✅ PASSED" if passed else "⚠️ ISSUES"
    print(f"   {status}")

    if not passed:
        all_passed = False

print(f"\n🎯 FINAL SUMMARY")
print("=" * 50)

# Processing statistics
total_raw = len(raw_dataset)
total_processed = sum(len(df) for df in processed_datasets.values())
failed_datasets = len(progressive_loader.failed_datasets)

print(f"📈 Processing Results:")
print(f"   Raw samples collected: {total_raw:,}")
print(f"   Quality samples processed: {total_processed:,}")
print(f"   Processing efficiency: {(total_processed/total_raw*100) if total_raw > 0 else 0:.1f}%")
print(f"   Failed datasets: {failed_datasets}")
print(f"   Memory cleanups performed: {progressive_loader.processing_stats['memory_cleanups']}")

print(f"\n🎵 Audio Processing:")
for rate_category, df in processed_datasets.items():
    sample_rate = config.sample_rates[rate_category]
    duration = df['processed_duration'].sum() / 3600 if 'processed_duration' in df.columns else 0
    print(f"   {rate_category} ({sample_rate}Hz): {len(df):,} samples, {duration:.1f}h")

print(f"\n📋 Model Support:")
total_models = sum(len(models) for models in config.model_categories.values())
total_manifests = sum(v['manifest_models'] for v in validation_results.values())
print(f"   Models supported: {total_models}")
print(f"   Manifests created: {total_manifests}")
print(f"   Manifest success rate: {(total_manifests/total_models*100) if total_models > 0 else 0:.1f}%")

print(f"\n🚀 Next Steps:")
print(f"   1. ✅ Data processing complete - all systems ready")
print(f"   2. 🔄 Update existing training notebooks to use new manifests")
print(f"   3. 🚀 Start training models with processed data")
print(f"   4. 📊 Monitor training progress and quality metrics")
print(f"   5. 🔧 Fine-tune based on initial results")

# Final status
if all_passed:
    print(f"\n🎉 SUCCESS: All validation checks PASSED!")
    print(f"✅ Enhanced resilient data processing completed successfully")
    print(f"🚀 Ready for model training with high-quality processed data")
else:
    print(f"\n⚠️ PARTIAL SUCCESS: Some validation checks failed")
    print(f"✅ Core processing completed with usable data")
    print(f"🔧 Review validation results and address any issues")

print("\n" + "=" * 70)
print("🏆 ENHANCED RESILIENT DATA PROCESSING COMPLETE! 🏆")
print("=" * 70)